# LoRA Task Switching — GPT-2 with Two Adapters
### Extending *LoRA: Low-Rank Adaptation of Large Language Models* (Hu et al., 2021)

**Core idea:** Train two separate LoRA adapters on completely different tasks.  
Load both onto the **same frozen GPT-2**. Switch behavior with a single line of code.

```
Base GPT-2 (frozen, 124M params — never changes)
    ├── Adapter 1: trained on TinyStories  →  generates creative fiction
    └── Adapter 2: trained on AG News      →  generates news-style text
```

> *"We can switch tasks by replacing the matrices A and B, reducing storage  
> and task-switching overhead significantly."* — Hu et al., 2021

## 1 · Setup

In [ ]:
!pip install torchao --upgrade -q
!pip install transformers peft datasets accelerate evaluate scikit-learn ipywidgets -q
print('Installation complete.')

In [ ]:
import os, time, warnings
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset

warnings.filterwarnings('ignore')
os.makedirs('./adapters/stories', exist_ok=True)
os.makedirs('./adapters/news', exist_ok=True)
print('Libraries imported.')

## 2 · Configuration

In [ ]:
CONFIG = {
    'model_name'   : 'gpt2',       # GPT-2 small (124M params)
    'max_length'   : 256,
    'batch_size'   : 8,
    'epochs'       : 3,
    'lr'           : 3e-4,
    'lora_r'       : 8,
    'lora_alpha'   : 16,
    'lora_dropout' : 0.05,
    'lora_targets' : ['c_attn'],   # GPT-2 combined Q,K,V projection matrix
    'stories_n'    : 2000,
    'news_n'       : 2000,
    'seed'         : 42,
}
set_seed(CONFIG['seed'])
for k, v in CONFIG.items():
    print(f'  {k:<18}: {v}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# GPT-2 has no pad token by default — set it to eos_token
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer loaded. Vocab size: {tokenizer.vocab_size:,}')

## 3 · Load Datasets

In [ ]:
# Adapter 1: TinyStories — short imaginative stories for children
stories_raw = load_dataset(
    'roneneldan/TinyStories',
    split=f'train[:{CONFIG["stories_n"]}]'
)
print(f'Stories : {len(stories_raw):,} examples')
print(f'Sample  : {stories_raw[0]["text"][:150]}...')

In [ ]:
# Adapter 2: AG News — real-world news articles
news_raw = load_dataset(
    'ag_news',
    split=f'train[:{CONFIG["news_n"]}]'
)
print(f'News   : {len(news_raw):,} examples')
print(f'Sample : {news_raw[0]["text"][:150]}...')

## 4 · Tokenization

In [ ]:
def tokenize_dataset(dataset, text_field='text'):
    def _tok(batch):
        return tokenizer(
            batch[text_field],
            truncation=True,
            max_length=CONFIG['max_length'],
            padding=False,
        )
    remove_cols = [c for c in dataset.column_names if c != text_field]
    ds = dataset.map(_tok, batched=True, remove_columns=remove_cols)
    ds.set_format('torch')
    return ds

stories_ds = tokenize_dataset(stories_raw, 'text')
news_ds    = tokenize_dataset(news_raw,    'text')

# DataCollatorForLanguageModeling handles causal LM labels automatically
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print('Tokenization complete.')
print(f'Stories columns : {stories_ds.column_names}')
print(f'News columns    : {news_ds.column_names}')

## 5 · LoRA Setup

In [ ]:
def make_lora_model():
    '''Load fresh GPT-2 and inject LoRA matrices.'''
    base = AutoModelForCausalLM.from_pretrained(CONFIG['model_name'])
    base.resize_token_embeddings(len(tokenizer))

    lora_cfg = LoraConfig(
        task_type      = TaskType.CAUSAL_LM,
        r              = CONFIG['lora_r'],
        lora_alpha     = CONFIG['lora_alpha'],
        target_modules = CONFIG['lora_targets'],
        lora_dropout   = CONFIG['lora_dropout'],
        bias           = 'none',
    )
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()
    return model

def get_train_args(output_dir):
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = CONFIG['epochs'],
        per_device_train_batch_size = CONFIG['batch_size'],
        learning_rate               = CONFIG['lr'],
        fp16                        = torch.cuda.is_available(),
        save_strategy               = 'no',
        report_to                   = 'none',
        logging_steps               = 50,
        seed                        = CONFIG['seed'],
    )

print('LoRA helpers defined.')

## 6 · Train Adapter 1 — Stories

Fine-tune LoRA on **TinyStories**: short, simple, imaginative stories.  
The adapter learns narrative structure, character descriptions, and creative language.

In [ ]:
print('Training Adapter 1: Stories...\n')

stories_model = make_lora_model()

trainer1 = Trainer(
    model         = stories_model,
    args          = get_train_args('./adapters/stories_ckpt'),
    train_dataset = stories_ds,
    data_collator = data_collator,
)

t0 = time.time()
trainer1.train()
stories_time = time.time() - t0

print(f'\nAdapter 1 done in {stories_time:.0f}s ({stories_time/60:.1f} min)')

In [ ]:
# Save only the LoRA adapter weights — not the full model
stories_model.save_pretrained('./adapters/stories')

# Show how small the adapter file is
def adapter_size_mb(path):
    total = 0
    for f in os.listdir(path):
        if 'adapter_model' in f:
            total += os.path.getsize(os.path.join(path, f))
    return total / 1024**2

stories_mb = adapter_size_mb('./adapters/stories')
print(f'Adapter 1 saved   : ./adapters/stories')
print(f'Adapter file size : {stories_mb:.2f} MB')
print(f'Files             : {os.listdir("./adapters/stories")}')

del stories_model
torch.cuda.empty_cache()

## 7 · Train Adapter 2 — News

Fine-tune a second LoRA adapter on **AG News**: real-world news articles.  
The adapter learns factual, concise, journalistic writing style.

In [ ]:
print('Training Adapter 2: News...\n')

news_model = make_lora_model()

trainer2 = Trainer(
    model         = news_model,
    args          = get_train_args('./adapters/news_ckpt'),
    train_dataset = news_ds,
    data_collator = data_collator,
)

t0 = time.time()
trainer2.train()
news_time = time.time() - t0

print(f'\nAdapter 2 done in {news_time:.0f}s ({news_time/60:.1f} min)')

In [ ]:
news_model.save_pretrained('./adapters/news')

news_mb = adapter_size_mb('./adapters/news')
print(f'Adapter 2 saved   : ./adapters/news')
print(f'Adapter file size : {news_mb:.2f} MB')

del news_model
torch.cuda.empty_cache()

## 8 · Task Switching

**This is the key demonstration.**

One base GPT-2 model. Both adapters loaded simultaneously.  
A single `set_adapter()` call flips between completely different behavior.  
The frozen base weights **never change**.

In [ ]:
# Load base model ONCE
base_model = AutoModelForCausalLM.from_pretrained(CONFIG['model_name'])
base_model.resize_token_embeddings(len(tokenizer))

# Attach BOTH adapters to the same base model object
switch_model = PeftModel.from_pretrained(
    base_model, './adapters/stories', adapter_name='stories'
)
switch_model.load_adapter('./adapters/news', adapter_name='news')
switch_model = switch_model.to(device)
switch_model.eval()

print('Both adapters loaded on one model.')
print(f'Available adapters : {list(switch_model.peft_config.keys())}')
print()
print('To switch tasks:')
print('  switch_model.set_adapter("stories")  # creative fiction')
print('  switch_model.set_adapter("news")     # news writing')

In [ ]:
def generate(prompt, adapter, max_new_tokens=150, temperature=0.85):
    switch_model.set_adapter(adapter)   # ← this is the entire task switch
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = switch_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            do_sample          = True,
            top_p              = 0.92,
            repetition_penalty = 1.3,
            pad_token_id       = tokenizer.eos_token_id,
        )
    # Slice off the prompt tokens — return only new generated text
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print('generate() ready.')

In [ ]:
PROMPTS = [    'The old lighthouse had been abandoned for years, but last night',    'Scientists have discovered that',    'The weather today is',]print('=' * 68)print('   TASK SWITCHING DEMO — Same Prompt, Different Adapters')print('=' * 68)for prompt in PROMPTS:    out_stories = generate(prompt, 'stories')    out_news    = generate(prompt, 'news')    print(f'\nPROMPT: "{prompt}"')    print(f'\n  📖 [Stories Adapter]')    print(f'  {out_stories[:300]}')    print(f'\n  📰 [News Adapter]')    print(f'  {out_news[:300]}')    print('─' * 68)

## 9 · Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('LoRA Task Switching — Two Adapters on One Base Model',
             fontsize=13, fontweight='bold')

# Storage comparison
gpt2_full_mb = 548.0   # GPT-2 small full model ~548 MB
labels  = ['Stories\nAdapter', 'News\nAdapter', 'Full GPT-2\n(per task copy)']
sizes   = [stories_mb, news_mb, gpt2_full_mb]
colors  = ['#4CAF50', '#2196F3', '#FF5722']

bars = axes[0].bar(labels, sizes, color=colors, width=0.5,
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Storage Per Task (MB)', fontweight='bold')
axes[0].set_ylabel('Megabytes')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
for bar, v in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 v + 8, f'{v:.1f} MB',
                 ha='center', fontweight='bold', fontsize=10)
axes[0].set_ylim(0, gpt2_full_mb * 1.3)

# Training time
axes[1].bar(['Stories\nAdapter', 'News\nAdapter'],
            [stories_time/60, news_time/60],
            color=['#4CAF50', '#2196F3'],
            width=0.5, edgecolor='white', linewidth=1.5)
axes[1].set_title('Adapter Training Time', fontweight='bold')
axes[1].set_ylabel('Minutes')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
for i, v in enumerate([stories_time/60, news_time/60]):
    axes[1].text(i, v + 0.05, f'{v:.1f} min',
                 ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('./adapter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./adapter_comparison.png')

## 10 · Interactive DemoUse the widget below to try any prompt with either adapter.  Switch the adapter toggle to see how the same prompt generates completely different text.

In [ ]:
import ipywidgets as widgetsfrom IPython.display import displaytitle = widgets.HTML(    "<h3>🔀 LoRA Task Switching Demo</h3>"    "<p>Same frozen GPT-2 base model &mdash; two LoRA adapters &mdash; one line to switch</p>")prompt_box = widgets.Textarea(    value='The old lighthouse had been abandoned for years, but last night',    description='Prompt:',    layout=widgets.Layout(width='100%', height='80px'))adapter_toggle = widgets.RadioButtons(    options=['📖 Stories Adapter', '📰 News Adapter'],    value='📖 Stories Adapter',    description='Adapter:',)max_tokens_slider = widgets.IntSlider(    value=150, min=50, max=300, step=10,    description='Max Tokens:',    layout=widgets.Layout(width='420px'))temperature_slider = widgets.FloatSlider(    value=0.85, min=0.1, max=1.5, step=0.05,    description='Temperature:',    layout=widgets.Layout(width='420px'))generate_btn = widgets.Button(    description='Generate ▶',    button_style='success',    layout=widgets.Layout(width='160px'))output_box = widgets.Textarea(    value='',    description='Output:',    layout=widgets.Layout(width='100%', height='200px'))status = widgets.HTML("")def on_generate(b):    status.value = "<i>Generating...</i>"    adapter = 'stories' if 'Stories' in adapter_toggle.value else 'news'    result = generate(        prompt_box.value,        adapter,        max_tokens_slider.value,        temperature_slider.value    )    output_box.value = result    status.value = f"<b>Done.</b> Active adapter: <code>{adapter}</code>"generate_btn.on_click(on_generate)display(title, prompt_box, adapter_toggle,        max_tokens_slider, temperature_slider,        generate_btn, status, output_box)